# Setup

In [ ]:
!pip install spuco

In [2]:
import torch

device = torch.device("cuda")

In [3]:
from spuco.utils import set_seed

set_seed(0)

# Dataset Creation

In [ ]:
from spuco.robust_train import ERM
from spuco.datasets import SpuCoMNIST, SpuriousFeatureDifficulty
import torchvision.transforms as T

classes = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]
difficulty = SpuriousFeatureDifficulty.MAGNITUDE_LARGE

trainset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    spurious_correlation_strength=0.995,
    classes=classes,
    split="train",
    label_noise=0.001
)
trainset.initialize()

testset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    classes=classes,
    split="test"
)
testset.initialize()


In [5]:
from spuco.datasets import SpuCoMNIST, SpuriousFeatureDifficulty

classes = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]

valset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    classes=classes,
    split="val"
)
valset.initialize()


100%|██████████| 11996/11996 [00:01<00:00, 7308.78it/s]


In [6]:
T.ToPILImage()(trainset[1000][0]).resize((28,28))

# Train using ERM

In [7]:
from spuco.models import model_factory

model = model_factory("lenet", trainset[0][0].shape, trainset.num_classes).to(device)

In [8]:
from torch.optim import SGD

erm = ERM(
    model=model,
    num_epochs=3,
    trainset=trainset,
    batch_size=64,
    optimizer=SGD(model.parameters(), lr=1e-2, momentum=0.9, nesterov=True),
    device=device,
    verbose=True
)

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [9]:
erm.train()

Epoch 2: 100%|██████████| 751/751 [00:09<00:00, 75.34batch/s, accuracy=100.0%, loss=0.00793]


In [10]:
from spuco.evaluate import Evaluator

evaluator = Evaluator(
    testset=testset,
    group_partition=testset.group_partition,
    group_weights=trainset.group_weights,
    batch_size=64,
    model=model,
    device=device,
    verbose=True
)
evaluator.evaluate()

Evaluating group-wise accuracy:   4%|▍         | 1/25 [00:00<00:05,  4.20it/s]

Group (0, 0) Accuracy: 100.0


Evaluating group-wise accuracy:   8%|▊         | 2/25 [00:00<00:05,  4.24it/s]

Group (0, 1) Accuracy: 0.0


Evaluating group-wise accuracy:  12%|█▏        | 3/25 [00:00<00:05,  4.31it/s]

...

Evaluating group-wise accuracy:  96%|█████████▌| 24/25 [00:09<00:00,  1.98it/s]

Group (4, 3) Accuracy: 0.0


Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:10<00:00,  2.50it/s]

Group (4, 4) Accuracy: 100.0


{(0, 0): 100.0,
 (0, 1): 0.0,
 (0, 2): 0.0,
 (0, 3): 0.0,
 (0, 4): 0.0,
 (1, 0): 0.0,
 (1, 1): 100.0,
 (1, 2): 0.0,
 (1, 3): 0.0,
 (1, 4): 0.0,
 (2, 0): 0.0,
 (2, 1): 0.0,
 (2, 2): 100.0,
 (2, 3): 0.0,
 (2, 4): 0.0,
 (3, 0): 0.0,
 (3, 1): 0.0,
 (3, 2): 0.0,
 (3, 3): 100.0,
 (3, 4): 0.0,
 (4, 0): 0.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (4, 3): 0.0,
 (4, 4): 100.0}

In [11]:
evaluator.worst_group_accuracy

((0, 1), 0.0)

In [12]:
evaluator.average_accuracy

99.39380051662361

In [13]:
evaluator.evaluate_spurious_attribute_prediction()

100.0

# Train using JTT

## Group Inference

In [16]:
# Evaluate all groups for Question 1 and extract accuracies for specific groups
group_accuracies = evaluator.evaluate()

# Extract accuracies for groups (1,1) and (0,1)
accuracy_group_1_1 = group_accuracies.get((1,1))
accuracy_group_0_1 = group_accuracies.get((0,1))

print("Accuracy for Group (1,1):", accuracy_group_1_1)
print("Accuracy for Group (0,1):", accuracy_group_0_1)

Evaluating group-wise accuracy:   4%|▍         | 1/25 [00:00<00:05,  4.32it/s]

Group (0, 0) Accuracy: 100.0


Evaluating group-wise accuracy:   8%|▊         | 2/25 [00:00<00:05,  4.23it/s]

Group (0, 1) Accuracy: 0.0


Evaluating group-wise accuracy:  12%|█▏        | 3/25 [00:00<00:05,  4.26it/s]

...

Group (4, 2) Accuracy: 0.0


Evaluating group-wise accuracy:  96%|█████████▌| 24/25 [00:05<00:00,  4.25it/s]

Group (4, 3) Accuracy: 0.0


Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:05<00:00,  4.25it/s]

Group (4, 4) Accuracy: 100.0
Accuracy for Group (1,1): 100.0
Accuracy for Group (0,1): 0.0


In [17]:
# Check the size of groups for Question 2 to analyze class imbalance
print("Size of Group (1,1):", len(testset.group_partition[(1,1)]))
print("Size of Group (0,1):", len(testset.group_partition[(0,1)]))

# Print accuracies again if needed for further analysis
print("Accuracy for Group (1,1):", accuracy_group_1_1)
print("Accuracy for Group (0,1):", accuracy_group_0_1)

Size of Group (1,1): 409
Size of Group (0,1): 423
Accuracy for Group (1,1): 100.0
Accuracy for Group (0,1): 0.0


In [18]:
# Function to reinitialize the train and test sets with a specified spurious correlation strength
def reinitialize_datasets(spurious_correlation_strength):
    global trainset, testset

    trainset = SpuCoMNIST(
        root="/data/mnist/",
        spurious_feature_difficulty=difficulty,
        spurious_correlation_strength=spurious_correlation_strength,
        classes=classes,
        split="train",
        label_noise=0.001
    )
    trainset.initialize()

    testset = SpuCoMNIST(
        root="/data/mnist/",
        spurious_feature_difficulty=difficulty,
        classes=classes,
        split="test"
    )
    testset.initialize()

# Loop through different spurious correlation strengths and evaluate group (0,1) accuracy
strengths = [0.995, 0.975, 0.9]
results = {}

for strength in strengths:
    reinitialize_datasets(spurious_correlation_strength=strength)

    # Re-instantiate the model and train again
    model = model_factory("lenet", trainset[0][0].shape, trainset.num_classes).to(device)
    optimizer = SGD(model.parameters(), lr=1e-2, momentum=0.9, nesterov=True)
    erm = ERM(
        model=model,
        num_epochs=3,
        trainset=trainset,
        batch_size=64,
        optimizer=optimizer,
        device=device,
        verbose=True
    )
    erm.train()

    # Re-evaluate after training
    evaluator = Evaluator(
        testset=testset,
        group_partition=testset.group_partition,
        group_weights=trainset.group_weights,
        batch_size=64,
        model=model,
        device=device,
        verbose=True
    )
    group_accuracies = evaluator.evaluate()

    # Store accuracy of group (0,1) for the current spurious correlation strength
    accuracy_group_0_1 = group_accuracies.get((0,1))
    results[strength] = accuracy_group_0_1
    print(f"Spurious Correlation Strength {strength}: Accuracy of Group (0,1): {accuracy_group_0_1}")

Evaluating group-wise accuracy:   4%|▍         | 1/25 [00:00<00:07,  3.10it/s]

Group (0, 0) Accuracy: 100.0


Evaluating group-wise accuracy:   8%|▊         | 2/25 [00:00<00:07,  2.94it/s]

Group (0, 1) Accuracy: 0.0


Evaluating group-wise accuracy:  12%|█▏        | 3/25 [00:00<00:07,  3.05it/s]

...

Group (4, 2) Accuracy: 56.423173803526446


Evaluating group-wise accuracy:  96%|█████████▌| 24/25 [00:10<00:00,  2.41it/s]

Group (4, 3) Accuracy: 88.38383838383838


Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:11<00:00,  2.24it/s]

Group (4, 4) Accuracy: 99.74747474747475
Spurious Correlation Strength 0.9: Accuracy of Group (0,1): 96.92671394799055


In [19]:
# Define a function to update the dataset with varying spurious feature difficulty
def set_spurious_difficulty(difficulty):
    global trainset, testset

    trainset = SpuCoMNIST(
        root="/data/mnist/",
        spurious_feature_difficulty=difficulty,
        spurious_correlation_strength=0.995,
        classes=classes,
        split="train",
        label_noise=0.001
    )
    trainset.initialize()

    testset = SpuCoMNIST(
        root="/data/mnist/",
        spurious_feature_difficulty=difficulty,
        classes=classes,
        split="test"
    )
    testset.initialize()

# Set different difficulty levels and track worst-group accuracy
difficulties = [
    SpuriousFeatureDifficulty.MAGNITUDE_SMALL,
    SpuriousFeatureDifficulty.MAGNITUDE_MEDIUM,
    SpuriousFeatureDifficulty.MAGNITUDE_LARGE
]
worst_group_accuracies = {}

for difficulty in difficulties:
    # Set dataset with the specified spurious difficulty
    set_spurious_difficulty(difficulty)

    # Re-instantiate the model and trainer
    model = model_factory("lenet", trainset[0][0].shape, trainset.num_classes).to(device)
    optimizer = SGD(model.parameters(), lr=1e-2, momentum=0.9, nesterov=True)
    erm = ERM(
        model=model,
        num_epochs=3,
        trainset=trainset,
        batch_size=64,
        optimizer=optimizer,
        device=device,
        verbose=True
    )
    erm.train()

    # Re-evaluate and store the worst-group accuracy
    evaluator = Evaluator(
        testset=testset,
        group_partition=testset.group_partition,
        group_weights=trainset.group_weights,
        batch_size=64,
        model=model,
        device=device,
        verbose=True
    )
    evaluator.evaluate()
    worst_group_accuracies[difficulty] = evaluator.worst_group_accuracy

# Print results for each spurious feature difficulty
for difficulty, accuracy in worst_group_accuracies.items():
    print(f"Spurious Feature Difficulty {difficulty}: Worst-group Accuracy = {accuracy}")

Evaluating group-wise accuracy:   4%|▍         | 1/25 [00:00<00:13,  1.74it/s]

Group (0, 0) Accuracy: 99.05437352245863


Evaluating group-wise accuracy:   8%|▊         | 2/25 [00:01<00:11,  2.05it/s]

Group (0, 1) Accuracy: 99.52718676122932


Evaluating group-wise accuracy:  12%|█▏        | 3/25 [00:01<00:10,  2.11it/s]

...

Group (4, 2) Accuracy: 0.0


Evaluating group-wise accuracy:  96%|█████████▌| 24/25 [00:12<00:00,  1.51it/s]

Group (4, 3) Accuracy: 0.0


Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:12<00:00,  1.97it/s]

Group (4, 4) Accuracy: 100.0
Spurious Feature Difficulty SpuriousFeatureDifficulty.MAGNITUDE_SMALL: Worst-group Accuracy = ((2, 0), 94.93333333333334)
Spurious Feature Difficulty SpuriousFeatureDifficulty.MAGNITUDE_MEDIUM: Worst-group Accuracy = ((4, 1), 14.6095717884131)
Spurious Feature Difficulty SpuriousFeatureDifficulty.MAGNITUDE_LARGE: Worst-group Accuracy = ((0, 1), 0.0)


## Robust Training

In [21]:
# Set the spurious correlation strength and spurious feature difficulty as required
set_spurious_difficulty(SpuriousFeatureDifficulty.MAGNITUDE_LARGE)
trainset.spurious_correlation_strength = 0.995
trainset.initialize()
testset.initialize()

# Step 1: Train the model to obtain initial predictions
from spuco.models import model_factory
from spuco.utils import Trainer
from torch.optim import SGD

# Create and train an initial model
model = model_factory("mlp", trainset[0][0].shape, trainset.num_classes).to(device)
trainer = Trainer(
    trainset=trainset,
    model=model,
    batch_size=64,
    optimizer=SGD(model.parameters(), lr=1e-2, momentum=0.9, nesterov=True),
    device=device,
    verbose=True
)
trainer.train(1)

# Obtain predictions for group inference
predictions = torch.argmax(trainer.get_trainset_outputs(), dim=-1).detach().cpu().tolist()

# Step 2: Use JTT's Group Inference with predictions and true labels
from spuco.group_inference import JTTInference

jtt_inference = JTTInference(
    predictions=predictions,
    class_labels=trainset.labels
)
group_partition = jtt_inference.infer_groups()

# Step 3: Perform robust training with JTT's UpsampleERM method using the inferred group labels
from spuco.robust_train import UpSampleERM

jtt_trainer = UpSampleERM(
    model=model,
    num_epochs=10,
    trainset=trainset,
    batch_size=64,
    group_partition=group_partition,
    optimizer=SGD(model.parameters(), lr=1e-2, weight_decay=5e-4, momentum=0.9, nesterov=True),
    device=device,
    verbose=True
)
jtt_trainer.train()

# Step 4: Evaluate model performance after robust training, focusing on worst-group accuracy
from spuco.evaluate import Evaluator

evaluator = Evaluator(
    testset=testset,
    group_partition=testset.group_partition,
    group_weights=trainset.group_weights,
    batch_size=64,
    model=model,
    device=device,
    verbose=True
)
evaluator.evaluate()

# Print the worst-group accuracy achieved after JTT training
print("Worst-group accuracy after JTT training:", evaluator.worst_group_accuracy)

Evaluating group-wise accuracy:   4%|▍         | 1/25 [00:00<00:14,  1.64it/s]

Group (0, 0) Accuracy: 96.6903073286052


Evaluating group-wise accuracy:   8%|▊         | 2/25 [00:01<00:12,  1.78it/s]

Group (0, 1) Accuracy: 84.16075650118204


Evaluating group-wise accuracy:  12%|█▏        | 3/25 [00:01<00:12,  1.81it/s]

...

Group (4, 2) Accuracy: 0.0


Evaluating group-wise accuracy:  96%|█████████▌| 24/25 [00:13<00:00,  1.78it/s]

Group (4, 3) Accuracy: 0.0


Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:14<00:00,  1.75it/s]

Group (4, 4) Accuracy: 0.0
Worst-group accuracy after JTT training: ((1, 3), 0.0)


# Download Notebook as PDF


In [ ]:
!apt-get install texlive texlive-xetex texlive-latex-extra pandoc
!pip install pypandoc

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
!cp "/content/drive/My Drive/Colab Notebooks/my_cs260d_hw3.ipynb" ./

In [38]:
!jupyter nbconvert --to PDF "my_cs260d_hw3.ipynb"

[NbConvertApp] Converting notebook my_cs260d_hw3.ipynb to PDF
[NbConvertApp] Support files will be in my_cs260d_hw3_files/
[NbConvertApp] Making directory ./my_cs260d_hw3_files
[NbConvertApp] Writing 120181 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | bibtex had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 81698 bytes to my_cs260d_hw3.pdf
